# 00 — Environment + BF16 baseline (LFM3-1.7B)

Colab Pro, **A100-40GB runtime** (Runtime → Change runtime type → A100).

This notebook:
1. Mounts Google Drive at `/content/drive` and clones/pulls the repo into `MyDrive/lfm3-ternary` so `results/` persists across sessions.
2. Installs pinned deps: `transformers>=5.12.1`, `lm-eval==0.4.7`, `accelerate`, `datasets`.
3. Loads `LiquidAI/LFM3-1.7B` in BF16 with `trust_remote_code=True` and sanity-generates.
4. Runs `scripts/eval_lm_eval.sh` (PIQA, ARC-E, ARC-C, HellaSwag, Winogrande; zero-shot; seed 2) → `results/baseline.json`.

**Secrets:** add `HF_TOKEN` in the Colab 🔑 sidebar (or `export HF_TOKEN=...` before launching).

## 1. Drive mount + repo checkout

The repo lives inside Drive so `results/baseline.json` survives session expiry. `cd`'s into the repo and stays there for the rest of the notebook.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/drive/MyDrive/lfm3-ternary"
REPO_URL = os.environ.get("LFM3_TERNARY_REPO_URL", "https://github.com/ORG/LFM3-Ternary.git")
BRANCH = "feat/lfm3-ternary"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    assert "ORG" not in REPO_URL, "set LFM3_TERNARY_REPO_URL to this repo's clone URL"
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}

os.chdir(REPO_DIR)
print("repo:", os.getcwd())
!git log -1 --oneline

## 2. Dependencies

In [ ]:
!pip install -q "transformers>=5.12.1" "lm-eval==0.4.7" accelerate datasets

## 3. Hugging Face auth

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass  # not on Colab or no secret; HF_TOKEN env var still honored

assert os.environ.get("HF_TOKEN"), "set HF_TOKEN in Colab secrets or the environment"
!huggingface-cli whoami | head -3

## 4. Load BF16 model + sanity generation

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "LiquidAI/LFM3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
model.eval()
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 5. Baseline eval → `results/baseline.json`

Runs the pinned five-task lm-eval set. ~30–60 min on A100. The JSON this writes is the baseline every rung's `gates.py` verdict compares against.

In [ ]:
os.makedirs("results", exist_ok=True)
!bash scripts/eval_lm_eval.sh "{MODEL_ID}" results/baseline.json

In [ ]:
import json

baseline = json.load(open("results/baseline.json"))
print(baseline)
assert sorted(baseline) == ["arc_challenge", "arc_easy", "hellaswag", "piqa", "winogrande"]
assert all(0.0 < v <= 1.0 for v in baseline.values())
print("baseline.json OK — synced back to Drive automatically")